# Kokoro TTS Android - Cloud Compiler
Run this notebook in **Google Colab** to build the Kokoro TTS APK in ~2 minutes.

### Step 1: Install OpenJDK 17 & Android SDK

In [ ]:
# 1. Install JDK 17
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk

# 2. Install Android Command-Line Tools & SDK 34
!mkdir -p /root/Android/Sdk/cmdline-tools
!cd /root/Android/Sdk/cmdline-tools && \
  curl -s -O https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip && \
  unzip -q -o commandlinetools-linux-*.zip && \
  rm -f commandlinetools-linux-*.zip && \
  rm -rf latest && \
  mv cmdline-tools latest

# 3. Accept Android Licenses and install platforms
!yes | /root/Android/Sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null
!/root/Android/Sdk/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null
print("✅ Android SDK 34 is successfully installed!")

### Step 2: Upload KokoroTTS-Android.zip

In [ ]:
import os
from google.colab import files

if not os.path.exists("/content/KokoroTTS-Android.zip"):
    print("Please upload your KokoroTTS-Android.zip file:")
    uploaded = files.upload()
else:
    print("✅ KokoroTTS-Android.zip is already present in /content/")

### Step 3: Unpack and Compile the APK

In [ ]:
%cd /content
!rm -rf project && mkdir project
!unzip -q -o KokoroTTS-Android.zip -d project
!echo "sdk.dir=/root/Android/Sdk" > project/local.properties
!chmod +x project/gradlew

%cd /content/project
# Run assembleDebug (this compiles the arm64-v8a APK)
!export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 && ./gradlew assembleDebug

# Copy generated APKs to /content for easy access
!cp /content/project/app/build/outputs/apk/debug/*.apk /content/ 2>/dev/null || true

import glob
apks = glob.glob('/content/*.apk')
if apks:
    print("\n🎉 SUCCESS! Generated APK(s):")
    for a in apks:
        print(f"  -> {a} ({os.path.getsize(a) / (1024*1024):.1f} MB)")
else:
    print("\n❌ Build did not produce an APK. Check the log above for errors.")

### Step 4: Download the Generated APK

In [ ]:
import glob
from google.colab import files

apks = glob.glob("/content/*.apk")
if apks:
    for apk in apks:
        print(f"Downloading {apk} to your computer...")
        files.download(apk)
    print("\nIf your browser blocks the automatic download popup:")
    print("Click the 📁 Files icon on the left sidebar in Colab, find the .apk file, click the 3 dots, and select 'Download'!")
else:
    print("No APK files found to download. Please run Step 3 successfully first.")